# Chapter 7

In the previous chapters, we analysed *the amplitude of the signal over time (**ERPs**)*. In this chapter, we will explore the **spectral content** of the data - *breaking the signal down into its componenet frequencies (oscillations) to understand "brain rhythms" like Alpha, Beta, and Gamma*.
<br />
We will perform two main analysis:
1. **PSD (Power Spectral Density):** A static overview of "how much" of each frequencies exists in the data.
2. **TFR (Time-Frequency Representation):** A dynamic view showing when these frequencies changes power over time.

> **Pro Tip: Energy (Power) = Amplitude squared ($A^2$):** <br />
> So if the *Alpha wave (8-12 Hz)* becomes "taller" (higher peaks and deeper troughs on the graph), that means it has **higher amplitude**, which translates directly to **higher energy/power**.
> + *Low Alpha Power:* The neurons are barely whispering in the Alpha rhythm.
> + *High Alpha Power:* The neurons are shouting in the Alpha rhythm.

> **Pro Tip: Why Time-Frequency Analysis?**<br />
> Standard **ERP analysis (averaging trails)** is great for seeing *phase-locked activity (activity that happens at the exacty same time and phase in every trail)*. However, the brain also produces **induced** (succeed in persuading or leading (someone) to do something, or bring about or give rise to.) activity.
>   + *Example: Imagine a crowd clapping.*
>       - **ERP:** Everyone claps at the *exact same millisecond*. You hear a loud "BANG".
>       - **Induced (Oscillatory):** Everyone claps for 5 seconds, but not in perfect sync. If you average the sound waves, they cancel out to silence. But if you look at the 
**Power (Energy)**, you see a huge increase in "clapping energy".
>   + *Goal: Time-Frequency analysis lets us see this "clapping energy" (Power) even if the waves aren't perfectly synced.*

> **The Brain Bands:** <br />
>   + **Delta (1-4 Hz):** Deep sleep, or very slow cognitive processes.
>   + **Theta (4-8 Hz):** Often related to *memory encoding/retrieval and cognitive control* (like doing a difficult math problem).
>   + **Alpha (8-12 Hz):** The most famous rhythm! It usually reflects *inhibition* (
a feeling that makes one self-conscious and unable to act in a relaxed and natural way) or *idling* :
>       - *High Alpha:* Brain area is "shutting down" / resting (e.g., visual cortex when eyes are closed).
>       - *Low Alpha:* Brain area is active / processing.
>   + **Beta (12-30 Hz):** associated with *motor control* (movement planning) and active concentration/alertness.
>   + **Gamma (>30 Hz):** High-level feature binding and conscious processing.

## Libraries and Config


In [ ]:
import pathlib
import matplotlib
import numpy as np

import mne
import mne_bids

matplotlib.use("QtAgg")
mne.set_log_level("WARNING")

## Loading Data
We will start by loading the epochs and apply the pre-equipped *SSP projectos* because frequency analysis often requires clean data, so applying SSP projectors (`epochs.apply_proj()`) is crucial here to remove heartbeats and blinks that could contaminate the spectrum. We will also only focus on `Auditory` trials:

In [ ]:
# load the epochs
epochs = mne.read_epochs(
    pathlib.Path("out_data") / "epochs-epo.fif"
)

# clean the epochs
epochs.apply_proj()

# extract the auditory trails
epochs_auditory = epochs["Auditory"]

epochs_auditory

## Frequency Analysis (Static Power Spectral Density)
Before looking at time, let's just look at *frequency*. Standard *ERP analysis* averages out *non-phase-locked activity*. The **Power Spectral Density (PSD)** tells us the average energy of different brain rhythms across the entire epochs, i.e., it lets us see the "background hum" of the brain. Luckily, the `epochs` instance comes with `.plot_psd()` and the peaks in the plot usually correspond to canonical brain bands: **Theta (4-8Hz), Alpha (8-12Hz), or Beta (13-30Hz)**.

> **Pro Tip: Understanding `bandwidth` param** <br />
>   The `bandwidth=2` parameter controls the "smoothness" of the spectral estimation (using Multitapers method).
>   + Higher bandwidth: Smoother plot, but might blur two close peaks together (e.g., 10Hz and 12Hz become one blob).
>   + Lower bandwidth: sharper peaks, but "noisier" looking.

> **Phase-Locked (Evoked) vs Non-Phase-Locked (Induced)** <br />
When you record brain data (EEG), you repeat an experiment many times (e.g., 100 trails of hearing a beep).
>   + **Phase-Locked (Evoked):** Activity that happens at the exact same time and in the same direction (peak or trough) in every single trial.
>   + **Non-Phase-Locked (Induced):** Activity that happends in every trail (e.g., a burst of energy), but the peaks and troughs don't line up perfectly.
>
> *Example:* Imagine a football stadium with 100 fans (trails). You ask them all to clap their hands once when you blow a whistle.
>   + *Scenario A: Perfect Synchronisation (Phase-Locked / ERP)*
>       - *Instruction:* Clap exactly 1 second after the whistle.
>       - *Result:* Everyone claps at the exact same moment.
>       - *Average:* If you average the sound, you get a huge *BANG* at 1 second.
>       - This is your standard ERP.
>
>   + *Scenario B: Jittered Timing (Non-Phase-Locked / Induced)*
>       - *Instruction:* Clap sometime between 1 and 2 seconds after the whistle.
>       - *Result:* Everyone claps, but at slightly different times. One person claps at 1.1s, another at 1.5s, another at 1.9s.
>       - *Average:* If you average the sound waveform directly, the claps cancel each other out. One person's clap peak overlaps with another person's silence. The average is neraly flat (silence).
>       - Standard ERP analysis misses this entirely.
>
>   + *The Solution:*
>       - *Method:* Instead of averagin the sound wave, we calculate the energy (loudness) for each person first.
>       - *Result:* We see that every person generate a loudness of 80 decibles between 1s and 2s.
>       - *Average:* The average energy is high. We now know the crowd was active, even though they weren't synchronised.

In [ ]:
# plot average PSD across all auditory epochs
epochs_auditory.plot_psd(
    fmin=2., # minimum frequency to plot
    fmax=40., # maximum frequency to plot
    average=True, # average across epochs
    bandwidth=2.0, # smoothing bandwidth
)

Let's now visualise the **Spatail Distribution**, i.e., where on the head these frequencies are strongest using topomaps. We use the `.plot_psd_topomap()` func of the `epochs` instance and the main two parameters it takes are `ch_type=` and `normalise=`.

> **Note: Normalisation** <br />
>   + `normalize=False:` Shows absolute power/PSD ( $V^2/Hz$ ). Good for seeing raw strength.
>   + `normalize=True:` Shows realtive power (percentage of total energy). Good for seeing if a specific rhythm is dominant, even if the oveerall signal is weak.

In [ ]:
epochs_auditory.plot_psd_topomap(
    ch_type="eeg",
    normalize=False
)

epochs_auditory.plot_psd_topomap(
    ch_type="mag",
    normalize=False
)

epochs_auditory.plot_psd_topomap(
    ch_type="grad",
    normalize=False
)

In [ ]:
epochs_auditory.plot_psd_topomap(
    ch_type="eeg",
    normalize=True
)

epochs_auditory.plot_psd_topomap(
    ch_type="mag",
    normalize=True
)

epochs_auditory.plot_psd_topomap(
    ch_type="grad",
    normalize=True
)

## Time-Frequency Analysis (Dynamics)

Now to know when (time) the frequency power changes, we can perform **Time-Frequency Analysis**. We use the `tfr_morlet()` method from `mne.time_frequency`.
<br />
This uses **"Morlet Wavelets"**. *Think of a wavelet as a small, rhythmic "template" or "probe". We slide this template across you data (**convolution**\*)*.
+ *Match:* If the brain signal matches the template's rhythm at a specific moment, the output value is high, i.e., if the brain signal oscillates at the same frequency as the wavelet, the output spikes (High Power).
+ *No Match:* If the brain signal is flat, chaotic or doesn't match the rhythm, the output is low or zero.
<br />
<br />
The function returns two objects
1. **Power:** The strength of the oscillation (e.g., The Alpha rhythm got louder). This capture *Induced Activity*.
2. **ITC (Inter-Trail Coherence):** The phase consistency (e.g., The peaks aligned perfectly across all 100 trails -  this captures Evoked Activity.).
<br />
<br />
Critical Parameters of `tfr_morlet()`:
1. `freqs` (*Log-spaced*): <br />
We usually space frequencies logarithmically (2, 4, 8, 16 ...) rather than linearly (2, 3, 4, 5, ...). This is because physiological bands wider at higher frequencies (Gamma is braod range, Delta is a narrow range).

2. `n_cycles` (*The Trade-off*): <br />
This is the most important parameter. It controls the **Heisenberg Uncertainty Principle of signal processing**.
    + *High `n_cycles` (e.g., 10):* Great Frequency Resolution (you can tell 10Hz from 11Hz perfectly), but terrible Time Resolution (you don't know if it happened at 100ms or 300ms, it's smeared).
    + *Low `n_cycles` (e.g., 2):* Great Time Resolution (you see exactly when it happened), but poor Frequency Resolution (10Hz blurs into 15Hz).
    + *The "Variable Cycles" Strategy (`freqs / 2`):* Instead of using a fixed number of cycles for everything, we adapt:
        - At low freqs (2Hz):, We use fewer cycles. Since low waves are naturally very long (slow), using fewer cycles helps us keep the time window from becoming distinctively huge.
        - At high freqs (30Hz): We use more cycles. Since high waves are very short (fast), we can afford to use more cycles to get better frequency precision without losing much time information.

> _**Convolution** is a mathematical operation that combines two functions (or signals/images) to create a third, showing how the shape of one modifies the other, essentially acting as a "sliding filter" to extract features like edges in images or determine system outputs from inputs in signal processing._

<div style="text-align: center;">
<img src="./imgs/wavelet_convolution.gif" alt="Wavelet Convolution example" width="250"/>
<br />
*Source: <a href="https://www.youtube.com/shorts/ykJVzqG2fpM">Youtube</a>
</div>

In [ ]:
from mne.time_frequency import tfr_morlet

# create a list with 20 values ranging between 2, 30 with logspace
freqs = np.logspace(
    *np.log10([2, 30]),
    num=20
)

n_cycles = freqs / 2

# Compute Time-Frequency Representation (TFR) using Morlet wavelets
power, itc = tfr_morlet(
    epochs_auditory, 
    freqs=freqs,
    n_cycles=n_cycles,
    use_fft=True, # Use FFT for faster computation
    return_itc=True,
    decim=3, # Decimate: Downsample to save memory (we don't need 1000Hz sampling for TFR)
    n_jobs=-1
)

### Handling Edge Artifacts
Wavelet convolution creates "edge artifacts" at the very beginning and very end of the data. This happens because the wavelet "tempalte" hangs off the edge of the data, trying to analyze zeros or missing values.
<br />
<br />
We simply cut these artifacts off using `.crop()`.

In [ ]:
# Crop to keep only the clean central part (e.g., -0.1s to 0.7s) 
power.crop(
    tmin=-0.1,
    tmax=0.7
)

itc.crop(
    tmin=-0.1,
    tmax=0.7
)

## Baseline Correction and Visualisation

### The "Pink Noise" Problem ($1/f$ Law)
Raw brain signals follow a $1/f$ distribution (Pink Noise). This means:

+ **Low Frequencies (e.g., Delta 2Hz):** Have naturally *massive amplitude* (huge waves, or like a jet engine).

+ **High Frequencies (e.g., Gamma 40Hz):** Have naturally *tiny* amplitude (microscopic ripples, or like a whisper).

If we plot the raw power, the massive low frequencies would dominate the color scale, creating a big red blob at the bottom and hiding all the subtle high-frequency activity.

### The Solution: Baseline Correction
To fix this, we normalize the data. Instead of looking at absolute power ("How loud is it?"), we look at relative change ("Did it get louder compared to silence?").
<br />
<br />
We compare the post-stimulus activity to a "quiet" pre-stimulus baseline period.

+ `baseline = (None, 0)`: This tells MNE to use all data from the start of the epoch up until time t=0 (the stimulus onset) as the "quiet period."

+ `mode = 'logratio'`: This calculates the change in Decibels (dB).
    - *Value = 0 (Green):* No change from baseline.

    - *Positive (Red):* Power increased (Synchronization).

    - *Negative (Blue):* Power decreased (Desynchronization).

In [ ]:
baseline_mode = "logratio"  # Calculates: 10 * log10(Active / Baseline) -> Output in dB

baseline = (None, 0) # Use pre-stimulus period (start -> 0s) as reference

### Visualizing Brain Rhythms in Space (Topomaps)
We can now plot topographic maps to see where on the head these rhythms are changing.

Let's first simply plot the power from the eeg channel use the `plot_topo()` func:

In [ ]:
(power.copy().pick_types(eeg=True, meg=False).plot_topo(
    baseline=baseline,
    mode=baseline_mode,
    title="EEG Power (dB) - Auditory"
))

Let's try to plot a single channel using the `.plot()`:

In [ ]:
power.plot(
    picks="EEG 050",
    baseline=baseline,
    mode=baseline_mode
)

Now let's look at three canonical bands using the `.plot_topomap()` and passing the desired freq ranges using its `fmin` and `fmax` params:


In [ ]:
import matplotlib.pyplot as plt

fig, axis = plt.subplots(1, 4, figsize=(7, 4))

# Delta Band topomap
power.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=1.0, fmax=4.0, # frequency band of Delta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[0], # plot on first axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[0].set_title("Delta Band (1-4 Hz)")


# Theta Band topomap
power.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=4.0, fmax=7.0, # frequency band of Theta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[1], # plot on second axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[1].set_title("Theta Band (4-7 Hz)")


# Alpha Band topomap
power.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=8.0, fmax=12.0, # frequency band of Alpha
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[2], # plot on third axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[2].set_title("Alpha Band (8-12 Hz)")

# Beta Band topomap
power.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=13.0, fmax=30.0, # frequency band of Beta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[3], # plot on fourth axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[3].set_title("Beta Band (13-30 Hz)")

mne.viz.adjust_axes(axes=axis) # adjust spacing between subplots
plt.show()

Let's plot a **Joint Plot** using `.plot_joint`, as it provides a more comprehensive visualisation by combining two views:
1. *Main Heatmap:*  Shows the Time-Frequency spectrum averaged accross all sensors.
2. *Topomaps:* Shows the spatial distribution for specific "peaks" you selected from the heatmap. 

In [ ]:
# Plot the average TFR + Topomaps for interesting time-frequency points
power.plot_joint(
    baseline=baseline, mode="mean", # apply baseline correction with mean
    # Draw topomaps for (t=0.05s, f=2Hz), (t=0.07s, f=6Hz), (t=0.1s, f=10Hz), (t=0.15s, f=20Hz)
    timefreqs=[
        (0.05, 2),  # Delta
        (0.07, 6),   # Theta
        (0.1, 10),  # Alpha
        (0.15, 20)   # Beta 
    ]
)
plt.show()

***Baseline correction can also be applied directly to power using `power.apply_baseline()` with `basline=(None, 0)` and `mode="logratio"` params passed to it.***

Let's us now visualise the inter-trail coherence values (or phase consistency), using it `plot_topo()` and `plot_topomap()` and `plot_joint()` methods:

In [ ]:
itc.plot_topo(
    title="Inter-Trial coherence", 
    vmin=0., vmax=0.5, cmap='Reds' # color map settings 
)

In [ ]:
import matplotlib.pyplot as plt

fig, axis = plt.subplots(1, 4, figsize=(7, 4))

# Delta Band topomap
itc.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=1.0, fmax=4.0, # frequency band of Delta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[0], # plot on first axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[0].set_title("Delta Band (1-4 Hz)")


# Theta Band topomap
itc.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=4.0, fmax=7.0, # frequency band of Theta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[1], # plot on second axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[1].set_title("Theta Band (4-7 Hz)")


# Alpha Band topomap
itc.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=8.0, fmax=12.0, # frequency band of Alpha
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[2], # plot on third axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[2].set_title("Alpha Band (8-12 Hz)")

# Beta Band topomap
itc.plot_topomap(
    ch_type="grad",
    tmin=0.0, tmax=0.2, # time window of interest
    fmin=13.0, fmax=30.0, # frequency band of Beta
    baseline=baseline, mode=baseline_mode, # apply baseline correction
    axes=axis[3], # plot on fourth axis
    show=False,
    contours=1 # number of contour lines to draw
)
axis[3].set_title("Beta Band (13-30 Hz)")

mne.viz.adjust_axes(axes=axis) # adjust spacing between subplots
plt.show()

In [ ]:
itc.plot_joint(
    baseline=baseline, mode="mean", # apply baseline correction with mean
    # Draw topomaps for (t=0.05s, f=2Hz), (t=0.07s, f=6Hz), (t=0.1s, f=10Hz), (t=0.15s, f=20Hz)
    timefreqs=[
        (0.05, 2),  # Delta
        (0.07, 6),   # Theta
        (0.1, 10),  # Alpha
        (0.15, 20)   # Beta 
    ]
)
plt.show()

# The End & Have a great day!